# பாடம் 04 - கருவி பயன்பாட்டு வடிவமைப்பு மாதிரி

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework (Python) பயன்படுத்தி AI முகவர்களுக்கான **கருவி பயன்பாட்டு** வடிவமைப்பு மாதிரியை கற்றுக்கொள்ளப்போகிறீர்கள். நாங்கள் கவர்வது:

- `@tool` அலங்காரியுடன் மற்றும் typed அளவுருக்களுடன் செயல்பாட்டு கருவிகளை வரையறுத்தல்
- ஒவ்வொரு கருவியும் என்ன செய்கிறதோ என்பதை மாடல் அறிந்துகொள்ள கருவி வடிவமைப்புகளை வழங்குதல்
- `approval_mode` மூலம் கருவி செயல்பாட்டை கட்டுப்படுத்துதல்
- Pydantic மாதிரிகள் மற்றும் `response_format` மூலம் **ஒழுங்கமைக்கப்பட்ட வெளியீட்டை** வழங்குதல்

இந்த சூழல் என்பது இலக்குகளைத் தேடும், கிடைக்குமை சரிபார்க்கும் மற்றும் விமானப் தகவலைப் பெறும் **பயண முன்பதிவு முகவர்** ஆகும்.


## அமைப்பு


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool டெக்கரேட்டர் மூலம் கருவிகளை வரையறுத்தல்

`@tool` டெக்கரேட்டர் ஒரு சாதாரண Python செயல்பாட்டை ஏஜெண்ட் அழைக்கக்கூடிய கருவியாக மாற்றுகிறது.
முக்கிய கருத்துகள்:

- **டாக்ஸ்டிரிங்** அந்த கருவியின் விளக்கமாக மாடல் காணும்.
- **வகை குறிப்பு**(கள்) (`Annotated` உடன் விளக்கங்கள் உள்ளிட்டவை) கருவி ஸ்கீமாவை வரையறுக்கும்.
- `approval_mode` ஒவ்வொரு அழைப்பையும் செயல்படுத்துவதற்கு முன் பயனர் அனுமதி அளிக்க வேண்டும் என்பதைக் கட்டுப்படுத்துகிறது.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## பல கருவிகளுடன் முகவரியை உருவாக்குதல்

பயனர் கேள்விக்கு பதில் அளிக்க மாடல் எந்த கருவியை வேண்டுமானாலும் பயன்படுத்த முடியும் என்பதற்காக அனைத்து மூன்று கருவிகளையும் கிளையண்டுக்குக் கொண்டு செல்லுங்கள்.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = await provider.create_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## கருவிகளுடன் கட்டமைக்கப்பட்ட வெளியீடு

`response_format` ஐ Pydantic மாதிரியாக அமைப்பதன் மூலம், முகவர் சிறந்த வகைபடுத்தப்பட்ட JSON பொருளை திரும்ப அளிக்க மறுப்பதில்லை, சுதந்திர வடிவ உரை பதிலாக. இது கீழ்நிலை குறியீடு முடிவை நிரல்பூர்வமாக உபயோகிக்க வேண்டிய போது உதவுகிறது.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = await provider.create_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## கருவி அங்கீகாரம் கோட்பாடுகள்

`@tool` இல் உள்ள `approval_mode` அளவுரு கருவி அழைப்புகள் செயல்படும Jennera மனித அங்கீகாரம் தேவைப்படுகிறதா என்பதை கட்டுப்படுத்துகிறது:

| முறையை | நடத்தை |
|---|---|
| `"never_require"` | கருவி தானாக இயங்கும் — பயனர் உறுதிப்படுத்தல் தேவையில்லை. |
| `"always_require"` | ஒவ்வொரு அழைப்பும் செயல்படுவதற்கு முன் பயனர் அங்கீகாரம் பெற வேண்டும். |

பக்க விளைவுகள் உள்ள கருவிகளுக்கு (உதாரணம்: விமானத்தை முன்பதிவு செய்தல், கடன் அட்டையை சார்ஜ் செய்தல்) `"always_require"` பயன்படுத்தவும், எனவே மனிதர் செயலில் தொடரும்.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

##தில் சுருக்கம்

இந்த பாடத்தில் நீங்கள் கற்றுக்கொண்டது:

1. டைப் செய்யப்பட்ட அளவுருக்களையும் கருவி வரையறைக்கும் கருவி யாணிகளாக செயல்படும் விவரக்குறிப்புகளையும் கொண்ட `@tool` அலங்காரியைப் பயன்படுத்திக் **கருவிகளை வரையறுக்க**.
2. முகாமையாளர் சிக்கலான கேள்விகளுக்கு பதில் அளிக்க தொடர்ச்சியாக அழைக்க முடியும் போல பல கருவிகளை **ஒன்றாக இணைக்க**.
3. `response_format` ஆக Pydantic மாடலை வழங்குவது மூலம் **கட்டமைக்கப்பட்ட வெளியீட்டை** திரும்பச் செய்ய.
4. `approval_mode` மூலம் நுணுக்கமான செயல்பாடுகளுக்கான மனித ஒப்புதலை வலியுறுத்தி **கருவி அனுமதியை கட்டுப்படுத்த**.

இந்த மாதிரிகள் நம்பகமான, தயாரிப்பு-தொகுக்கப்படக்கூடிய முகாமையாளர்களை பாதுகாப்பாக வெளிப்புற அமைப்புகளுடன் தொடர்பு கொள்ளக் கூடியவாறு உருவாக்குவதற்கான அடிப்படையை அமைக்கின்றன.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**பிராரம்பிக்கை**:  
இந்த ஆவணம் AI மொழி மாற்ற சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் சரியானதாக இருக்க முயன்றாலும், தானாக செய்யப்பட்ட மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறான தகவல்கள் இருக்கக்கூடும் என்பதை கவனத்தில் கொள்ளவும். இதன் மூல மொழியிலுள்ள அசல் ஆவணம் அதிகாரபூர்வ ஆதாரமாக கருதப்பட வேண்டும். முக்கியமான தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பு பரிந்துரைக்கப்படுகிறது. இந்த மொழிபெயர்ப்பின் பயன்படுத்தலால் ஏற்படும் எந்தவொரு தவறான புரிதல்கள் அல்லது புரிதல் தவிர்ப்புகளுக்கும் நாம் பொறுப்பேற்கவில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
